# Skeletal muscle: base clustering (supporting)

Part of the **[Fig. 6 chapter](fig6.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{indir}npc.tsv'`  ·  _metadata_
- `f'{indir}L2/{ct}/5kCG_embed.h5ad'`  ·  _embedding h5ad_
- `f'{indir}L2/{ct}/100k3C_embed.h5ad'`  ·  _embedding h5ad_
- `f'{indir}L2/{ct}/5kCG100k3C_embed.h5ad'`  ·  _embedding h5ad_
- `f'{outdir}{ct}_mergeany_rocpr.npy'`  ·  _other_
- `f'{ENTEX_ROOT}/allclist.tsv'`  ·  _sc/pseudobulk mC (allc)_
- `'Mus-Skl/5kCG100k3C_embed.h5ad'`  ·  _embedding h5ad_


In [1]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [2]:
from glob import glob

import anndata
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import seaborn as sns
from ALLCools.clustering import *
from ALLCools.integration.seurat_class import SeuratIntegration
from ALLCools.plot import *
from sklearn.metrics import pairwise_distances, roc_auc_score
from sklearn.preprocessing import normalize

# mpl.use('agg')
mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

import warnings
warnings.filterwarnings("ignore")


In [3]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [4]:
indir = f'{ENTEX_ROOT}/clustering/merged/'
outdir = f'{ENTEX_ROOT}/analysis/pairwise_prediction/'


In [5]:
ct = 'Mus-Skl'


In [6]:
npc = pd.read_csv(f'{indir}npc.tsv', sep='\t', header=None, index_col=0, names=['npc_cg', 'npc_3c'])

In [7]:
adata_mc = anndata.read_h5ad(f'{indir}L2/{ct}/5kCG_embed.h5ad')
adata_3c = anndata.read_h5ad(f'{indir}L2/{ct}/100k3C_embed.h5ad')
adata_3c = adata_3c[adata_mc.obs.index].copy()


In [8]:
coord_base = 'tsne'
adata = anndata.read_h5ad(f'{indir}L2/{ct}/5kCG100k3C_embed.h5ad')
npc_cg, npc_3c = npc.loc[ct].values
labelany = np.load(f'{outdir}{ct}_mergeany_rocpr.npy', allow_pickle=True)
# labelboth = np.load(f'{outdir}{ct}_mergeboth_rocpr.npy', allow_pickle=True)
ds = 150/np.sqrt(adata.shape[0])


In [9]:
# clustering name
clustering_name = 'L1'

# ConsensusClustering
# Important factores
n_neighbors = 25
leiden_resolution = 0.5
# this parameter is the final target that limit the total number of clusters
# Higher accuracy means more conservative clustering results and less number of clusters
target_accuracy = 0.85
min_cluster_size = 50

# Other ConsensusClustering parameters
metric = 'euclidean'
consensus_rate = 0.8
leiden_repeats = 500
random_state = 0
train_frac = 0.5
train_max_n = 500
max_iter = 50
n_jobs = 32

# Dendrogram via Multiscale Bootstrap Resampling
nboot = 10000
method_dist = 'correlation'
method_hclust = 'average'

plot_type = 'static'

In [10]:
cc = ConsensusClustering(model=None,
                         n_neighbors=n_neighbors,
                         metric=metric,
                         min_cluster_size=min_cluster_size,
                         leiden_repeats=leiden_repeats,
                         leiden_resolution=leiden_resolution,
                         consensus_rate=consensus_rate,
                         random_state=random_state,
                         train_frac=train_frac,
                         train_max_n=train_max_n,
                         max_iter=max_iter,
                         n_jobs=n_jobs,
                         target_accuracy=target_accuracy)


In [11]:
cc.fit_predict(normalize(adata_3c.obsm[f'100k3C_pc{npc_3c}_seuratL2'], axis=1))
adata_3c.obs[f'100k3C_pc{npc_3c}_leiden_init'] = cc.label.copy()


In [12]:
cc.fit_predict(normalize(adata_mc.obsm[f'5kCG_u{npc_cg}_seuratL2'], axis=1))
adata_mc.obs[f'5kCG_u{npc_cg}_leiden_init'] = cc.label.copy()


In [13]:
for i, xx in enumerate([adata, adata_mc, adata_3c]):
    tmp = xx.obs.copy()

In [14]:
adata.obs['leiden_init_mc'] = adata_mc.obs[f'5kCG_u{npc_cg}_leiden_init'].copy()
adata.obs['leiden_init_3c'] = adata_3c.obs[f'100k3C_pc{npc_3c}_leiden_init'].copy()
# leiden_group_mc = [[0], [3,5,8,10,11,13,15,17,19], [1,2,4,6,7,9,14,18]]
# leiden_group_mc = [[2], [0,5,8,9],[1,3,4,6,7]]
leiden_group_mc = [[0],[3,5,8,10,11,13,15,16],[1,2,4,6,7,9,12,14]]
leiden_group_3c = [[8], [1,4,5,6,7,9], [0,2,3]]
leiden_map = {f'c{xx}':f'mc{i}' for i,x in enumerate(leiden_group_mc) for xx in x}
adata_mc.obs['leiden_group'] = adata.obs['leiden_init_mc'].map(leiden_map)
leiden_map = {f'c{xx}':f'3c{i}' for i,x in enumerate(leiden_group_3c) for xx in x}
adata_3c.obs['leiden_group'] = adata.obs['leiden_init_3c'].map(leiden_map)
adata.obs['leiden_group_mc'] = adata_mc.obs['leiden_group'].copy()
adata.obs['leiden_group_3c'] = adata_3c.obs['leiden_group'].copy()
adata.obs[['leiden_group_mc', 'leiden_group_3c']].value_counts().unstack()


In [15]:
adata.obs['group_mc_3c'] = adata.obs['leiden_group_mc'].astype(str) + '-' + adata.obs['leiden_group_3c'].astype(str)


In [16]:
adata.write_h5ad('Mus-Skl/5kCG100k3C_embed.h5ad')
adata_mc.write_h5ad('Mus-Skl/5kCG_embed.h5ad')
adata_3c.write_h5ad('Mus-Skl/100k3C_embed.h5ad')
